# 57 같은 카드의 공통 조건 원문 보완

56의 DB/추출 선택이 동결된 뒤 실행한다. 원질문 → 구조화 조회 → 조회 카드 안에서 vector+BM25 → 기존 프로필 재정렬까지가 A다. 구조화 결과0이면 원래 상품/발급사 범위에서 비구조화 검색을 한 번 수행한다.

B는 A의 카드·순위·근거를 모두 유지하고 같은 카드 안에서 `원질문 + 전월실적 한도 횟수 서비스 선택 제외 조건`으로 FTS/BM25를 조회한다. 제목/경로의 고정 조건 키워드로 걸러 기존 BGE로 정렬하고, 동점 chunk ID 사전순으로 최대2개 원문 구역을 통째로 추가한다. A최대5+B최대2=카드별 최대7, 전체64000bytes. parent bundle 하나의 원문청크 최대5는 바꾸지 않는다.

신규 임베딩/API0. 보완 근거 수가 증가하는 실험이며 동일 입력량 실험이 아니다. 원문 정답·gold fact ID·정답 페이지는 선택 함수에 제공하지 않는다. `.py`는 58과 공유할 추가 규칙 검증과 장시간 CPU 검색 재개만 소유한다. HTML은 결과를 확인하고 직접 작성한다.

In [ ]:
from pathlib import Path
import importlib.util,sys,json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'notebooks/scripts').is_dir())
spec=importlib.util.spec_from_file_location('runtime57_notebook',ROOT/'notebooks/scripts/57_sequential_evidence_runtime.py')
T=importlib.util.module_from_spec(spec);sys.modules[spec.name]=T;spec.loader.exec_module(T)
queries,plans,manifest=T.prepare(ROOT/'notebooks/57_sequential_condition_evidence.ipynb')
print('동결 질문',len(queries),'프로필',len(T.X.PROFILES))

In [ ]:
resources=T.run(queries,plans,manifest)
print(json.dumps(resources,ensure_ascii=False,indent=2))

## 원문 사실 판정 후 선택

검색 완료 자체를 개선으로 판단하지 않는다. 같은 고정 질문·원문 사실 단위로 A/B의 필요한 근거 회수, 완전근거 확보, 카드 범위, 다른 혜택 조건 연결/충돌을 수동 검토한다. 완전근거 개선과 중요 관계 오류0이 함께 확인되지 않으면 A를 유지한다. source fact 판정과 56 추출 조건 정답은 검색 함수의 입력에 들어가지 않는다. 이후 58의 생성 비교는 별도 단계다.